In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# !pip install sentence-transformers faiss-cpu numpy

In [ ]:
import sqlite3
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/skeptical-scholar")
DB_PATH = PROJECT_ROOT / "data" / "db" / "arxiv.db"
INDEX_PATH = PROJECT_ROOT / "data" / "processed" / "dense_index.index"
CHUNKS_PATH = PROJECT_ROOT / "data" / "processed" / "chunk_ids.npy"

INDEX_PATH.parent.mkdir(parents=True, exist_ok=True)

conn = sqlite3.connect(str(DB_PATH))
conn.row_factory = sqlite3.Row
cursor = conn.cursor()
cursor.execute("SELECT chunk_id, paper_id, section, text FROM chunks")
rows = cursor.fetchall()
conn.close()

chunks = [dict(row) for row in rows]
texts = [c["text"] for c in chunks]
chunk_ids = [c["chunk_id"] for c in chunks]

print(f"Loaded {len(chunks)} chunks")

In [ ]:
dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(embeddings)

faiss.write_index(index, str(INDEX_PATH))
np.save(str(CHUNKS_PATH), chunk_ids)

print(f"Index saved to: {INDEX_PATH}")
print(f"Chunk IDs saved to: {CHUNKS_PATH}")
print(f"Index size: {index.ntotal} vectors, {dim} dimensions")
print("Done! Download dense_index.index and chunk_ids.npy from Drive.")
